In [1]:
import pandas as pd
import numpy as np
import os
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

df1 = pd.read_csv('./data/CDC Diabetes Dataset.csv')

In [2]:
# Keep only non-diabetic (0) and diabetic (2)
df = df1[df1['Diabetes_012'].isin([0, 2])].copy()

# 1) columns for ARM
arm_cols_binary = [
    'HighBP','HighChol','CholCheck','Smoker','Stroke','HeartDiseaseorAttack',
    'PhysActivity','Fruits','Veggies','HvyAlcoholConsump',
    'AnyHealthcare','NoDocbcCost','DiffWalk','Sex'
]

arm_cols_ordinal = ['BMI','Age','GenHlth','PhysHlth','MentHlth','Education','Income']

# 2) binning
df_arm = df[arm_cols_binary + arm_cols_ordinal].copy()

df_arm['BMI_bin'] = pd.cut(df_arm['BMI'], bins=[0, 18.5, 25, 30, 100], 
                           labels=['BMI_Under','BMI_Normal','BMI_Over','BMI_Obese'])
df_arm['Age_bin'] = pd.cut(df_arm['Age'], bins=[0, 4, 7, 10, 14], 
                           labels=['Age_Young','Age_Mid','Age_Older','Age_Oldest'])

# self-reported days: 0 / 1-13 / 14-30 (common shreshold)
df_arm['PhysHlth_bin'] = pd.cut(df_arm['PhysHlth'], bins=[-1, 0, 13, 30],
                                labels=['PhysHlth_0','PhysHlth_1_13','PhysHlth_14_30'])
df_arm['MentHlth_bin'] = pd.cut(df_arm['MentHlth'], bins=[-1, 0, 13, 30],
                                labels=['MentHlth_0','MentHlth_1_13','MentHlth_14_30'])

# GenHlth: 1-5
df_arm['GenHlth_bin'] = df_arm['GenHlth'].map({
    1:'GenHlth_Excellent',2:'GenHlth_VeryGood',3:'GenHlth_Good',4:'GenHlth_Fair',5:'GenHlth_Poor'
})

# Education/Income interprete into binary
df_arm['Edu_bin'] = np.where(df_arm['Education'] >= 4, 'Edu_High', 'Edu_Low')
df_arm['Income_bin'] = np.where(df_arm['Income'] >= 6, 'Income_High', 'Income_Low')

# 3) binary columns: 1 -> column name
for c in arm_cols_binary:
    df_arm[c] = df_arm[c].where(df_arm[c] == 1, pd.NA)
    df_arm[c] = df_arm[c].replace(1, c)

# 4) final transaction dataset for ARM
item_cols = arm_cols_binary + ['BMI_bin','Age_bin','PhysHlth_bin','MentHlth_bin','GenHlth_bin','Edu_bin','Income_bin']
transactions = df_arm[item_cols]


In [3]:
# 每行转为item list
tx = transactions.apply(lambda row: [x for x in row.dropna().tolist()], axis=1).tolist()

rng = np.random.RandomState(42)
sample_size = min(30000, len(tx))
idx = rng.choice(len(tx), size=sample_size, replace=False)
tx_s = [tx[i] for i in idx]

te = TransactionEncoder()
te_ary = te.fit(tx_s).transform(tx_s, sparse=True)
df_onehot = pd.DataFrame.sparse.from_spmatrix(te_ary, columns=te.columns_)

freq = apriori(df_onehot, min_support=0.02, use_colnames=True, low_memory=True)
freq = freq.sort_values('support', ascending=False)

rules = association_rules(freq, metric='lift', min_threshold=1.2)
rules = rules.sort_values(['lift','confidence','support'], ascending=False)

rules.head(10)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
4839274,"frozenset({AnyHealthcare, GenHlth_Poor})","frozenset({Income_Low, CholCheck, PhysHlth_14_...",0.047533,0.052567,0.022500,0.473352,9.004795,1.0,0.020001,1.798988,0.933311,0.289948,0.444132,0.450690
4839249,"frozenset({Income_Low, CholCheck, PhysHlth_14_...","frozenset({AnyHealthcare, GenHlth_Poor})",0.052567,0.047533,0.022500,0.428028,9.004795,1.0,0.020001,1.665233,0.938270,0.289948,0.399483,0.450690
4200140,frozenset({GenHlth_Poor}),"frozenset({Income_Low, CholCheck, PhysHlth_14_...",0.050733,0.052567,0.023867,0.470434,8.949277,1.0,0.021200,1.789074,0.935732,0.300462,0.441052,0.462230
4200115,"frozenset({Income_Low, CholCheck, PhysHlth_14_...",frozenset({GenHlth_Poor}),0.052567,0.050733,0.023867,0.454027,8.949277,1.0,0.021200,1.738668,0.937543,0.300462,0.424847,0.462230
3971825,frozenset({GenHlth_Poor}),"frozenset({Income_Low, PhysHlth_14_30, DiffWalk})",0.050733,0.054067,0.024433,0.481603,8.907580,1.0,0.021690,1.824728,0.935181,0.304023,0.451973,0.466757
3971816,"frozenset({Income_Low, PhysHlth_14_30, DiffWalk})",frozenset({GenHlth_Poor}),0.054067,0.050733,0.024433,0.451911,8.907580,1.0,0.021690,1.731958,0.938476,0.304023,0.422619,0.466757
5831847,"frozenset({AnyHealthcare, GenHlth_Poor})","frozenset({CholCheck, PhysHlth_14_30, Smoker, ...",0.047533,0.049067,0.020767,0.436886,8.903935,1.0,0.018434,1.688706,0.931991,0.273846,0.407831,0.430060
5831824,"frozenset({CholCheck, PhysHlth_14_30, Smoker, ...","frozenset({AnyHealthcare, GenHlth_Poor})",0.049067,0.047533,0.020767,0.423234,8.903935,1.0,0.018434,1.651391,0.933494,0.273846,0.394450,0.430060
4668438,"frozenset({AnyHealthcare, GenHlth_Poor})","frozenset({Income_Low, PhysHlth_14_30, DiffWalk})",0.047533,0.054067,0.022867,0.481066,8.897643,1.0,0.020297,1.822839,0.931907,0.290432,0.451405,0.452000
4668433,"frozenset({Income_Low, PhysHlth_14_30, DiffWalk})","frozenset({AnyHealthcare, GenHlth_Poor})",0.054067,0.047533,0.022867,0.422935,8.897643,1.0,0.020297,1.650535,0.938344,0.290432,0.394136,0.452000


In [4]:
strong_rules = rules[
    (rules["lift"] > 2.0) &
    (rules["confidence"] > 0.4)
]

strong_rules[["antecedents", "consequents", "support", "confidence", "lift"]]

,antecedents,consequents,support,confidence,lift
4839274,"frozenset({AnyHealthcare, GenHlth_Poor})","frozenset({Income_Low, CholCheck, PhysHlth_14_...",0.022500,0.473352,9.004795
4839249,"frozenset({Income_Low, CholCheck, PhysHlth_14_...","frozenset({AnyHealthcare, GenHlth_Poor})",0.022500,0.428028,9.004795
4200140,frozenset({GenHlth_Poor}),"frozenset({Income_Low, CholCheck, PhysHlth_14_...",0.023867,0.470434,8.949277
4200115,"frozenset({Income_Low, CholCheck, PhysHlth_14_...",frozenset({GenHlth_Poor}),0.023867,0.454027,8.949277
3971825,frozenset({GenHlth_Poor}),"frozenset({Income_Low, PhysHlth_14_30, DiffWalk})",0.024433,0.481603,8.907580
...,...,...,...,...,...
729990,"frozenset({Veggies, Age_Oldest, HighChol})","frozenset({Edu_High, Fruits, HighBP})",0.047467,0.489011,2.000045
5456969,"frozenset({Smoker, Age_Oldest, HighBP, PhysHlt...","frozenset({AnyHealthcare, CholCheck, Edu_High,...",0.021367,0.509944,2.000043
5588437,"frozenset({AnyHealthcare, Sex, Age_Oldest, Inc...","frozenset({MentHlth_0, HighChol})",0.021167,0.562943,2.000035
5578422,"frozenset({Smoker, Age_Oldest, GenHlth_Good})","frozenset({AnyHealthcare, HighBP, MentHlth_0})",0.021200,0.558875,2.000031
